# Transforming the country groups dataset

This notebook transforms the cleaned country groups dataset `ibe010-migration-by-country-groups-2016-2025-interim.csv` into an analysis-ready wide format dataset. The goal is to pivot `Rādītāji` column into two, add a new column named `Migrācijas saldo` (`Iebrauca Latvijā - Izbrauca no Latvijas`) and save the dataset in `data/processed` for further work.

In [1]:
import pandas as pd

country_groups_data = pd.read_csv("../data/interim/ibe010-migration-by-country-groups-2016-2025-interim.csv")

country_groups_data.head()

,Rādītāji,Valstu grupa,Gads,Skaits
0,Iebrauca Latvijā,Pavisam,2016,8345
1,Iebrauca Latvijā,ES-27 no 2020. g.,2016,2254
2,Iebrauca Latvijā,ES-28,2016,4780
3,Iebrauca Latvijā,ES kandidātvalstis,2016,32
4,Iebrauca Latvijā,EBTA,2016,185


The required libraries are imported and the cleaned country groups dataset is loaded.
The first rows confirm that the dataset was loaded as intended from the cleaning work, with all expected columns and values.

In [2]:
country_groups_pivoted = country_groups_data.pivot_table(
    index=["Valstu grupa", "Gads"],
    columns="Rādītāji",
    values="Skaits",
    aggfunc="sum"
).reset_index()

country_groups_pivoted.columns.name = None

country_groups_pivoted.head()

,Valstu grupa,Gads,Iebrauca Latvijā,Izbrauca no Latvijas
0,EBTA,2016,185,918
1,EBTA,2017,179,900
2,EBTA,2018,246,1048
3,EBTA,2019,254,1221
4,EBTA,2020,222,1116


The dataset is transformed from long to wide format using `.pivot_table()`.
`Valstu grupa` and `Gads` are used as identifier columns, so each row represents one country group in one year. The values in `Rādītāji` column are converted into two separate columns that were the values of the column: `Iebrauca Latvijā` and `Izbrauca no Latvijas`. The migration counts from `Skaits` column are placed into these new columns.


`aggfunc="sum"` is used to aggregate values if more than one value exists for the same combination of country group, year, and migration indicator. It was previously confirmed that no duplicate rows exist, so it is kept as the default aggregation option if new entries are added to the dataset in the future.

`.reset_index()` moves `Valstu grupa` and `Gads` back into normal columns after pivoting.

`country_groups_pivoted.columns.name = None` removes `Rādītāji` column-axis name that pandas keeps above the new columns after pivoting. It does not remove the columns themselves or change the data values; it only removes the unnecessary column-axis label.

The displayed rows confirm that the dataset pivot was executed successfully.

In [3]:
country_groups_pivoted.shape

(70, 4)

The pivoted dataset contains 70 rows and 4 columns.
The rows represent every combination of seven country groups and ten years.
The four columns now are `Valstu grupa`, `Gads`, `Iebrauca Latvijā`, and `Izbrauca no Latvijas`.

In [4]:
country_groups_pivoted["Migrācijas saldo"] = (
    country_groups_pivoted["Iebrauca Latvijā"] - country_groups_pivoted["Izbrauca no Latvijas"])

country_groups_pivoted.head()

,Valstu grupa,Gads,Iebrauca Latvijā,Izbrauca no Latvijas,Migrācijas saldo
0,EBTA,2016,185,918,-733
1,EBTA,2017,179,900,-721
2,EBTA,2018,246,1048,-802
3,EBTA,2019,254,1221,-967
4,EBTA,2020,222,1116,-894


A new column, `Migrācijas saldo`, is calculated as immigration minus emigration: `Migrācijas saldo = Iebrauca Latvijā - Izbrauca no Latvijas`. A negative migration saldo means that more people left Latvia than arrived. A positive migration saldo means that more people arrived in Latvia than left.

In [5]:
country_groups_pivoted.to_csv(
    "../data/processed/ibe010-migration-by-country-groups-2016-2025-processed.csv",
    index=False,
    encoding="utf-8-sig"
)

print("The dataset has been saved")

The dataset has been saved


The processed wide format dataset is saved to the `data/processed` folder for use in analysis as the next step in the planned methodology.
`index=False` prevents pandas from writing its numeric row index as an extra column in the `.csv` file. `encoding="utf-8-sig"` writes a UTF-8 file with a byte order mark, which helps some programs, including Microsoft Excel, to correctly recognize the encoding and display Latvian characters.